# 模块六 · 监督学习核心算法

## 6.3 其他常见算法概述

**课时安排：约2课时**

除了线性回归和逻辑回归，监督学习还有许多重要算法。本节将概述四种经典算法的原理，并通过 Scikit-learn 进行对比实验。

---

### 本节目录

1. K近邻算法（K-NN）
2. 决策树
3. 支持向量机（SVM）
4. 朴素贝叶斯
5. 🧪 四种算法对比实验
6. 💡 练习题

---

## 1. K近邻算法（K-Nearest Neighbors, K-NN）

### 核心思想

> "近朱者赤，近墨者黑"

K-NN 是一种**懒惰学习（Lazy Learning）**算法——没有显式的训练过程。预测时，找到距离最近的 $K$ 个训练样本，由它们的标签投票决定预测结果。

### 算法步骤

1. 计算待预测样本与所有训练样本的距离
2. 找到距离最小的 $K$ 个近邻
3. **分类**：对 $K$ 个近邻的标签进行多数投票
4. **回归**：对 $K$ 个近邻的标签取平均

### 距离度量

最常用的是**欧氏距离（Euclidean Distance）**：

$$d(\mathbf{x}^{(a)}, \mathbf{x}^{(b)}) = \sqrt{\sum_{j=1}^{n}(x_j^{(a)} - x_j^{(b)})^2}$$

其他常用距离：
- **曼哈顿距离**：$d = \sum_{j=1}^{n}|x_j^{(a)} - x_j^{(b)}|$
- **闵可夫斯基距离**：$d = \left(\sum_{j=1}^{n}|x_j^{(a)} - x_j^{(b)}|^p\right)^{1/p}$（$p=2$ 即欧氏距离）

### K值的选择

| K 值 | 效果 |
|------|------|
| **K 太小**（如 K=1） | 对噪声敏感，容易过拟合 |
| **K 太大** | 决策边界过于平滑，容易欠拟合 |
| **K 适中**（经验值：$\sqrt{m}$） | 较好的平衡 |

**经验法则**：$K \approx \sqrt{m}$（$m$ 为训练样本数），通常取奇数避免投票平局。

### kd树简介

暴力搜索的时间复杂度 $O(mn)$，对大规模数据效率低。

**kd树（k-dimensional tree）**是一种空间划分数据结构：
- 将数据递归地沿不同维度中位数切分
- 查询时剪枝，避免检查所有节点
- 平均查询复杂度 $O(\log m)$，最坏 $O(m)$

---

## 2. 决策树（Decision Tree）

### 核心思想

决策树通过一系列**是/否问题**（对特征的阈值判断）递归地将数据划分，最终到达叶节点给出预测。

```
              [特征 x1 ≤ 3?]
              /            \
          [是]            [否]
           /                \
    [特征 x2 ≤ 5?]    [特征 x1 ≤ 7?]
      /        \         /        \
   [类别A]  [类别B]  [类别B]  [类别A]
```

### 划分标准

#### （1）信息增益（Information Gain）

基于**信息熵（Entropy）**衡量不纯度：

$$H(S) = -\sum_{k=1}^{K} p_k \log_2 p_k$$

其中 $p_k$ 是类别 $k$ 在集合 $S$ 中的比例。

信息增益 = 划分前的熵 - 划分后的加权熵：

$$\text{IG}(S, A) = H(S) - \sum_{v \in \text{Values}(A)} \frac{|S_v|}{|S|} H(S_v)$$

#### （2）信息增益比（Gain Ratio）

信息增益偏向取值多的特征，信息增益比进行归一化：

$$\text{GR}(S, A) = \frac{\text{IG}(S, A)}{\text{IV}(A)}$$

其中固有值 $\text{IV}(A) = -\sum_v \frac{|S_v|}{|S|} \log_2 \frac{|S_v|}{|S|}$

#### （3）基尼系数（Gini Index）

Scikit-learn 默认使用基尼系数：

$$\text{Gini}(S) = 1 - \sum_{k=1}^{K} p_k^2$$

- Gini 系数越小，纯度越高
- 计算比熵更高效（不需要对数）

### 剪枝（Pruning）

决策树容易**过拟合**，需要通过剪枝来控制复杂度：

| 剪枝策略 | 说明 |
|----------|------|
| **预剪枝** | 限制树的最大深度、叶节点最小样本数、最小信息增益等 |
| **后剪枝** | 先生成完整树，再自底向上删除不重要的子树 |

> Scikit-learn 中常用参数：`max_depth`, `min_samples_split`, `min_samples_leaf`

---

## 3. 支持向量机（Support Vector Machine, SVM）

### 核心思想

找到使两类样本之间的**间隔（margin）最大**的超平面：

$$\max_{\mathbf{w}, b} \frac{2}{\|\mathbf{w}\|} \quad \text{s.t.} \quad y^{(i)}(\mathbf{w}^T \mathbf{x}^{(i)} + b) \geq 1, \forall i$$

等价于：

$$\min_{\mathbf{w}, b} \frac{1}{2}\|\mathbf{w}\|^2 \quad \text{s.t.} \quad y^{(i)}(\mathbf{w}^T \mathbf{x}^{(i)} + b) \geq 1$$

### 关键概念

- **支持向量**：距超平面最近的样本点（间隔边界上的点），决定了超平面的位置
- **间隔**：两类之间的距离 $= \frac{2}{\|\mathbf{w}\|}$
- **最大间隔**：SVM 的目标，找到泛化能力最强的划分

### 软间隔与正则化

实际数据线性不可分时，引入**松弛变量** $\xi^{(i)} \geq 0$：

$$\min \frac{1}{2}\|\mathbf{w}\|^2 + C \sum_{i=1}^{m} \xi^{(i)}$$

- $C > 0$：正则化参数
- $C$ 大：要求严格分类（可能过拟合）
- $C$ 小：允许更多误分类（更好的泛化）

### 核技巧（Kernel Trick）

当数据非线性可分时，将数据映射到高维空间使其线性可分：

$$\phi: \mathbb{R}^n \to \mathbb{R}^d \quad (d \gg n)$$

核函数直接计算高维空间的内积，避免显式映射：

$$K(\mathbf{x}^{(a)}, \mathbf{x}^{(b)}) = \phi(\mathbf{x}^{(a)})^T \phi(\mathbf{x}^{(b)})$$

常用核函数：

| 核函数 | 公式 | 适用场景 |
|--------|------|----------|
| **线性核** | $K = \mathbf{x}^T \mathbf{x}'$ | 线性可分数据 |
| **RBF核（高斯核）** | $K = e^{-\gamma \|\mathbf{x}-\mathbf{x}'\|^2}$ | 通用，最常用 |
| **多项式核** | $K = (\mathbf{x}^T \mathbf{x}' + c)^d$ | 特定非线性模式 |

> 💡 RBF核是最常用的"万能"核函数，但需要调 $\gamma$ 参数。

---

## 4. 朴素贝叶斯（Naive Bayes）

### 核心思想

基于**贝叶斯定理**，利用先验概率计算后验概率：

$$P(y | \mathbf{x}) = \frac{P(\mathbf{x} | y) \cdot P(y)}{P(\mathbf{x})}$$

预测规则：

$$\hat{y} = \arg\max_y P(y) \prod_{j=1}^{n} P(x_j | y)$$

### 条件独立性假设

朴素贝叶斯的关键假设：**各特征条件独立**：

$$P(\mathbf{x} | y) = P(x_1, x_2, \dots, x_n | y) = \prod_{j=1}^{n} P(x_j | y)$$

> 这个假设在实际中几乎不成立（因此称为"朴素"），但在实践中效果 surprisingly 好！

### 常见变体

| 变体 | 特征分布假设 | 适用场景 |
|------|-------------|----------|
| **高斯朴素贝叶斯** | $P(x_j|y) \sim \mathcal{N}(\mu_{jk}, \sigma_{jk}^2)$ | 连续特征 |
| **多项式朴素贝叶斯** | $P(x_j|y) \sim \text{Multinomial}$ | 文本/计数特征 |
| **伯努利朴素贝叶斯** | $P(x_j|y) \sim \text{Bernoulli}$ | 二值特征 |

### 拉普拉斯平滑（Laplace Smoothing）

当某个特征值在训练中从未出现时，$P(x_j|y) = 0$，会导致整个概率为0。

**拉普拉斯平滑**：给所有计数加1：

$$P(x_j | y) = \frac{\text{count}(x_j, y) + 1}{\text{count}(y) + V}$$

其中 $V$ 是特征取值的总种类数。

> Scikit-learn 中对应参数 `alpha`（默认 `alpha=1.0`）。

### 四种算法对比总结

| 算法 | 类型 | 关键超参 | 优点 | 缺点 |
|------|------|----------|------|------|
| **K-NN** | 懒惰学习 | K值、距离度量 | 简单直观、无需训练 | 预测慢、高维灾难 |
| **决策树** | 模型驱动 | max_depth、min_samples | 可解释性强 | 易过拟合 |
| **SVM** | 间隔最大化 | C、核函数、γ | 高维表现好 | 大数据慢、难调参 |
| **朴素贝叶斯** | 概率模型 | alpha（平滑） | 快速、适合小数据 | 独立性假设强 |

---

## 5. 🧪 四种算法对比实验

我们将在相同的数据集上训练这四种算法，全面对比它们的性能。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, classification_report, 
                              confusion_matrix, ConfusionMatrixDisplay)
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# ========================
# 生成数据集
# ========================
np.random.seed(42)
X, y = make_classification(
    n_samples=1000, n_features=2, n_redundant=0,
    n_informative=2, n_clusters_per_class=1,
    class_sep=1.5, random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print(f"数据集大小: 训练集 {X_train.shape[0]} 样本, 测试集 {X_test.shape[0]} 样本")
print(f"特征维度: {X_train.shape[1]}")
print(f"类别分布: {np.bincount(y)}")

In [ ]:
# ========================
# 导入四种算法
# ========================
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

# 初始化模型
models = {
    'K-NN (K=5)': KNeighborsClassifier(n_neighbors=5),
    '决策树': DecisionTreeClassifier(max_depth=5, random_state=42),
    'SVM (RBF)': SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42),
    '朴素贝叶斯': GaussianNB(),
}

# ========================
# 训练 & 评估
# ========================
results = {}
for name, model in models.items():
    model.fit(X_train_s, y_train)
    y_pred = model.predict(X_test_s)
    acc = accuracy_score(y_test, y_pred)
    cv_scores = cross_val_score(model, X_train_s, y_train, cv=5, scoring='accuracy')
    results[name] = {
        'model': model, 'y_pred': y_pred, 'acc': acc, 'cv_mean': cv_scores.mean(), 'cv_std': cv_scores.std()
    }
    print(f"\n{'='*45}")
    print(f"  {name}")
    print(f"{'='*45}")
    print(f"  测试集准确率: {acc:.4f}")
    print(f"  5折交叉验证:  {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

In [ ]:
# ========================
# 性能对比汇总表
# ========================
print("\n" + "=" * 70)
print(f"{'算法':<15} {'测试准确率':>12} {'交叉验证均值':>14} {'交叉验证标准差':>14}")
print("=" * 70)
for name, res in results.items():
    print(f"{name:<15} {res['acc']:>12.4f} {res['cv_mean']:>14.4f} {res['cv_std']:>14.4f}")
print("=" * 70)

In [ ]:
# ========================
# 可视化1: 决策边界对比
# ========================
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

for idx, (name, res) in enumerate(results.items()):
    ax = axes[idx]
    model = res['model']
    h = 0.02
    x_min, x_max = X_test_s[:, 0].min() - 1, X_test_s[:, 0].max() + 1
    y_min, y_max = X_test_s[:, 1].min() - 1, X_test_s[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, alpha=0.3, cmap=plt.cm.RdYlBu)
    ax.contour(xx, yy, Z, colors='k', linewidths=0.3)
    ax.scatter(X_test_s[:, 0], X_test_s[:, 1], c=y_test, cmap=plt.cm.RdYlBu,
               edgecolors='k', s=20, alpha=0.7)
    ax.set_title(f'{name}\n准确率={res["acc"]:.3f}', fontsize=13)
    ax.set_xlabel('特征 1')
    ax.set_ylabel('特征 2')

plt.suptitle('四种算法决策边界对比', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ========================
# 可视化2: 准确率对比柱状图 + 交叉验证箱线图
# ========================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

names = list(results.keys())
accs = [results[n]['acc'] for n in names]
colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']

# 准确率柱状图
ax1 = axes[0]
bars = ax1.bar(names, accs, color=colors, alpha=0.8, edgecolor='black', linewidth=0.5)
ax1.set_ylim(0.7, 1.0)
ax1.set_ylabel('测试集准确率', fontsize=12)
ax1.set_title('四种算法准确率对比', fontsize=14)
ax1.grid(axis='y', alpha=0.3)
for bar, acc in zip(bars, accs):
    ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
             f'{acc:.4f}', ha='center', va='bottom', fontsize=10)

# 交叉验证箱线图
ax2 = axes[1]
cv_data = []
for name in names:
    cv = cross_val_score(results[name]['model'], X_train_s, y_train, cv=10, scoring='accuracy')
    cv_data.append(cv)

bp = ax2.boxplot(cv_data, labels=names, patch_artist=True)
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax2.set_ylabel('准确率', fontsize=12)
ax2.set_title('10折交叉验证分布', fontsize=14)
ax2.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ========================
# 可视化3: 混淆矩阵
# ========================
fig, axes = plt.subplots(2, 2, figsize=(10, 10))
axes = axes.flatten()

for idx, (name, res) in enumerate(results.items()):
    ax = axes[idx]
    cm = confusion_matrix(y_test, res['y_pred'])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['类别0', '类别1'])
    disp.plot(ax=ax, cmap='Blues', values_format='d')
    ax.set_title(f'{name}', fontsize=13)

plt.suptitle('四种算法混淆矩阵对比', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ========================
# K值对K-NN性能的影响
# ========================
k_range = range(1, 31)
k_scores = []
for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k)
    scores = cross_val_score(knn, X_train_s, y_train, cv=5, scoring='accuracy')
    k_scores.append(scores.mean())

best_k = list(k_range)[np.argmax(k_scores)]
print(f"最优K值: {best_k}, 对应交叉验证准确率: {max(k_scores):.4f}")

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(list(k_range), k_scores, 'b-o', markersize=5, linewidth=2)
ax.axvline(x=best_k, color='r', linestyle='--', alpha=0.7, label=f'最优 K={best_k}')
ax.set_xlabel('K 值', fontsize=12)
ax.set_ylabel('5折交叉验证准确率', fontsize=12)
ax.set_title('K值对K-NN性能的影响', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
# ========================
# 决策树深度对性能的影响
# ========================
depth_range = range(1, 21)
depth_scores_train = []
depth_scores_test = []
depth_scores_cv = []

for d in depth_range:
    dt = DecisionTreeClassifier(max_depth=d, random_state=42)
    dt.fit(X_train_s, y_train)
    depth_scores_train.append(dt.score(X_train_s, y_train))
    depth_scores_test.append(dt.score(X_test_s, y_test))
    cv = cross_val_score(dt, X_train_s, y_train, cv=5, scoring='accuracy')
    depth_scores_cv.append(cv.mean())

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(list(depth_range), depth_scores_train, 'b-o', markersize=4, label='训练集准确率', linewidth=1.5)
ax.plot(list(depth_range), depth_scores_test, 'r-s', markersize=4, label='测试集准确率', linewidth=1.5)
ax.plot(list(depth_range), depth_scores_cv, 'g-^', markersize=4, label='5折CV准确率', linewidth=1.5)
ax.set_xlabel('树最大深度', fontsize=12)
ax.set_ylabel('准确率', fontsize=12)
ax.set_title('决策树深度对性能的影响（过拟合分析）', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.show()

print("观察：深度增大 → 训练准确率上升（甚至达到1.0）")
print("        但测试准确率可能在某个点后下降 → 过拟合！")
print("        交叉验证可以帮助选择最优深度。")

In [ ]:
# ========================
# SVM 核函数对比
# ========================
kernels = {
    'Linear': SVC(kernel='linear', C=1.0, random_state=42),
    'RBF (γ=0.1)': SVC(kernel='rbf', C=1.0, gamma=0.1, random_state=42),
    'RBF (γ=1)': SVC(kernel='rbf', C=1.0, gamma=1.0, random_state=42),
    'RBF (γ=10)': SVC(kernel='rbf', C=1.0, gamma=10.0, random_state=42),
}

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

for idx, (kname, model) in enumerate(kernels.items()):
    ax = axes[idx]
    model.fit(X_train_s, y_train)
    y_pred = model.predict(X_test_s)
    acc = accuracy_score(y_test, y_pred)
    
    h = 0.02
    x_min, x_max = X_test_s[:, 0].min() - 1, X_test_s[:, 0].max() + 1
    y_min, y_max = X_test_s[:, 1].min() - 1, X_test_s[:, 1].max() + 1
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, alpha=0.3, cmap=plt.cm.RdYlBu)
    ax.contour(xx, yy, Z, colors='k', linewidths=0.3)
    ax.scatter(X_test_s[:, 0], X_test_s[:, 1], c=y_test, cmap=plt.cm.RdYlBu,
               edgecolors='k', s=20, alpha=0.7)
    ax.set_title(f'SVM: {kname}\n准确率={acc:.3f}', fontsize=13)

plt.suptitle('SVM 不同核函数的决策边界', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

print("\nγ 值越大 → 决策边界越复杂（可能过拟合）")
print("γ 值越小 → 决策边界越平滑（可能欠拟合）")

---

## 6. 💡 练习题

### 练习1：K-NN 的距离度量对比

分别使用欧氏距离、曼哈顿距离和闵可夫斯基距离（p=3）训练K-NN模型，对比分类准确率。可以使用 `KNeighborsClassifier(metric=...)` 参数。

### 练习2：决策树可解释性

训练一个决策树并使用 `sklearn.tree.plot_tree()` 可视化树结构。回答：
1. 树选择了哪些特征进行划分？
2. 根节点和叶节点的含义是什么？
3. 限制 `max_depth=3` 后树发生了什么变化？

### 练习3：朴素贝叶斯文档分类

使用 Scikit-learn 的 `fetch_20newsgroups` 数据集（选择4个类别），训练多项式朴素贝叶斯分类器：
1. 使用 `TfidfVectorizer` 进行文本向量化
2. 对比不同 `alpha` 值（0.1, 0.5, 1.0, 2.0）的效果
3. 打印分类报告

### 练习4：综合实验（选做）

在 `load_wine` 数据集上，对四种算法进行系统比较：
- 使用 5折交叉验证
- 对每种算法进行超参数调优（GridSearchCV）
- 绘制 ROC 曲线
- 分析各算法的优劣势

---

> 📌 **本节要点回顾**：
> - K-NN 是懒惰学习算法，K值选择影响偏差-方差平衡
> - 决策树通过信息增益/基尼系数选择最优划分，需要剪枝防过拟合
> - SVM 通过最大间隔寻找最优超平面，核技巧处理非线性
> - 朴素贝叶斯基于贝叶斯定理和条件独立性假设，计算快速
> - 不同算法有各自的适用场景，实践中需根据数据和问题选择